In [9]:
# Libraries
%load_ext autoreload
%autoreload 2

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_text
from src.tokenizer import CharTokenizer
from src.dataset import split_data, get_batch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Load data and build Vocab

text = load_text()

print("Total characters:", len(text))

print("\nText sample:")
print(text[:500])

tokenizer = CharTokenizer()

tokenizer.fit(text)

vocab_size = tokenizer.vocab_size

print("Vocabulary size:", vocab_size)

print("\nCharacter to ID mapping:")

for char, index in tokenizer.stoi.items():
    print(f"{repr(char):8} -> {index}")



Total characters: 1115394

Text sample:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
Vocabulary size: 65

Character to ID mapping:
'\n'     -> 0
' '      -> 1
'!'      -> 2
'$'      -> 3
'&'      -> 4
"'"      -> 5
','      -> 6
'-'      -> 7
'.'      -> 8
'3'      -> 9
':'      -> 10
';'      -> 11
'?'      -> 12
'A'      -> 13
'B'      -> 14
'C'      -> 15
'D'      -> 16
'E'      -> 17
'F'      -> 18
'G'      -> 19
'H'      -> 20
'I'      -> 21
'J'      -> 22
'K'      -> 23
'L'      -> 24
'M'      -> 25
'N'      -> 26
'O'      -> 27
'P'

In [16]:
# Encoding and Decoding

sample_text = "ROMEO:\nBut soft"

encoded = tokenizer.encode(sample_text)

decoded = tokenizer.decode(encoded)

print("Original text:")
print(repr(sample_text))

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(repr(decoded))

assert decoded == sample_text

token_ids = torch.tensor(
    tokenizer.encode(text),
    dtype=torch.long
)

print("Token IDs shape:", token_ids.shape)
print("Data type:", token_ids.dtype)

print("\nFirst 30 token IDs:")
print(token_ids[:30])

Original text:
'ROMEO:\nBut soft'

Encoded:
[30, 27, 25, 17, 27, 10, 0, 14, 59, 58, 1, 57, 53, 44, 58]

Decoded:
'ROMEO:\nBut soft'
Token IDs shape: torch.Size([1115394])
Data type: torch.int64

First 30 token IDs:
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43])


In [12]:
# Data Split

train_data, val_data, test_data = split_data(
    token_ids,
    train_ratio=0.9,
    val_ratio=0.05
)

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Testing tokens:", len(test_data))

print("\nSplit ratios:")

print(
    f"Train: {len(train_data) / len(token_ids):.2%}"
)

print(
    f"Validation: {len(val_data) / len(token_ids):.2%}"
)

print(
    f"Test: {len(test_data) / len(token_ids):.2%}"
)

Training tokens: 1003854
Validation tokens: 55770
Testing tokens: 55770

Split ratios:
Train: 90.00%
Validation: 5.00%
Test: 5.00%


In [13]:
# Batch

batch_size = 4
block_size = 16

device = "cpu"

In [14]:
# Save processed data

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save tokenized datasets
torch.save(
    train_data,
    PROCESSED_DIR / "train.pt"
)

torch.save(
    val_data,
    PROCESSED_DIR / "val.pt"
)

torch.save(
    test_data,
    PROCESSED_DIR / "test.pt"
)

# Save tokenizer vocabulary
with open(
    PROCESSED_DIR / "tokenizer.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        tokenizer.to_dict(),
        file,
        ensure_ascii=False,
        indent=4
    )

print("Processed data saved successfully!")

Processed data saved successfully!


In [15]:
# Summary

summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Tokens": [
        len(train_data),
        len(val_data),
        len(test_data)
    ],
    "Percentage": [
        len(train_data) / len(token_ids) * 100,
        len(val_data) / len(token_ids) * 100,
        len(test_data) / len(token_ids) * 100
    ]
})

display(summary)

print("Vocabulary size:", tokenizer.vocab_size)
print("Block size:", block_size)
print("Batch size:", batch_size)

,Split,Tokens,Percentage
0,Train,1003854,89.999946
1,Validation,55770,5.000027
2,Test,55770,5.000027


Vocabulary size: 65
Block size: 16
Batch size: 4
